# 06 - Backend Integration

## CyberForge AI - API-Ready Model Packaging

This notebook prepares models for backend integration:
- Model serialization in API-friendly formats
- Versioned model artifacts
- Backend-compatible inference endpoints
- Integration with mlService.js and ThreatService.js

### Backend Integration Points:
- `mlService.js` - Primary ML model interface
- `WebScraperAPIService.js` - Input data format
- `threatService.js` - Threat analysis outputs

In [ ]:
import json
import os
import time
import shutil
from pathlib import Path
from typing import Dict, List, Any
import joblib
import hashlib
import warnings
warnings.filterwarnings('ignore')

# Configuration
config_path = Path("notebook_config.json")
if not config_path.exists():
    config_path = Path("/home/user/app/notebooks/notebook_config.json")
with open(config_path) as f:
    CONFIG = json.load(f)

MODELS_DIR = Path(CONFIG["datasets_dir"]).parent / "models"
AGENT_DIR = MODELS_DIR.parent / "agent"
BACKEND_DIR = MODELS_DIR.parent / "backend_package"
BACKEND_DIR.mkdir(exist_ok=True)

print(f"✓ Configuration loaded")
print(f"✓ Backend package output: {BACKEND_DIR}")

## 1. Define Backend API Contracts

In [ ]:
class BackendAPIContracts:
    """
    Define API contracts matching backend services.
    Aligned with mlService.js and threatService.js
    """
    
    # Request format from WebScraperAPIService
    SCRAPER_INPUT = {
        'url': 'string',
        'security_report': {
            'is_https': 'boolean',
            'mixed_content': 'boolean',
            'insecure_cookies': 'boolean',
            'security_headers': 'object'
        },
        'network_requests': 'array',
        'console_logs': 'array',
        'page_content': 'string'
    }
    
    # Response format for mlService.js
    ML_RESPONSE = {
        'prediction': 'number',  # 0 or 1
        'confidence': 'number',  # 0.0 to 1.0
        'risk_level': 'string',  # critical, high, medium, low, info
        'model_name': 'string',
        'model_version': 'string',
        'inference_time_ms': 'number',
        'details': 'object'
    }
    
    # Response format for threatService.js
    THREAT_RESPONSE = {
        'threat_detected': 'boolean',
        'threat_type': 'string',
        'risk_score': 'number',
        'indicators': 'array',
        'recommended_action': 'string',
        'reasoning': 'string'
    }
    
    @classmethod
    def format_ml_response(cls, prediction: int, confidence: float, 
                           model_name: str, inference_time: float,
                           details: Dict = None) -> Dict:
        """Format response for mlService.js"""
        risk_level = (
            'critical' if confidence >= 0.9 else
            'high' if confidence >= 0.7 else
            'medium' if confidence >= 0.5 else
            'low' if confidence >= 0.3 else 'info'
        )
        
        return {
            'prediction': int(prediction),
            'confidence': float(confidence),
            'risk_level': risk_level,
            'model_name': model_name,
            'model_version': '1.0.0',
            'inference_time_ms': float(inference_time),
            'details': details or {}
        }
    
    @classmethod
    def format_threat_response(cls, detected: bool, threat_type: str,
                               score: float, indicators: List,
                               action: str, reasoning: str) -> Dict:
        """Format response for threatService.js"""
        return {
            'threat_detected': detected,
            'threat_type': threat_type,
            'risk_score': float(score),
            'indicators': indicators,
            'recommended_action': action,
            'reasoning': reasoning
        }

print("✓ Backend API Contracts defined")

## 2. Model Packager

In [ ]:
class ModelPackager:
    """
    Package models for backend deployment.
    Creates versioned, self-contained model artifacts.
    """
    
    def __init__(self, models_dir: Path, output_dir: Path):
        self.models_dir = models_dir
        self.output_dir = output_dir
        self.package_manifest = {'models': {}, 'version': '1.0.0'}
    
    def calculate_checksum(self, file_path: Path) -> str:
        """Calculate MD5 checksum for file integrity"""
        with open(file_path, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()
    
    def package_model(self, model_name: str, model_info: Dict) -> Dict:
        """Package a single model for backend"""
        model_type = model_info.get('best_model', 'random_forest')
        source_dir = self.models_dir / model_name
        dest_dir = self.output_dir / model_name
        dest_dir.mkdir(exist_ok=True)
        
        packaged_files = {}
        
        # Copy model file
        model_source = source_dir / f"{model_type}.pkl"
        if model_source.exists():
            model_dest = dest_dir / "model.pkl"
            shutil.copy(model_source, model_dest)
            packaged_files['model'] = {
                'path': str(model_dest.relative_to(self.output_dir)),
                'checksum': self.calculate_checksum(model_dest),
                'size_bytes': model_dest.stat().st_size
            }
        
        # Copy scaler
        scaler_source = source_dir / "scaler.pkl"
        if scaler_source.exists():
            scaler_dest = dest_dir / "scaler.pkl"
            shutil.copy(scaler_source, scaler_dest)
            packaged_files['scaler'] = {
                'path': str(scaler_dest.relative_to(self.output_dir)),
                'checksum': self.calculate_checksum(scaler_dest)
            }
        
        # Copy metadata
        meta_source = source_dir / f"{model_type}_metadata.json"
        if meta_source.exists():
            meta_dest = dest_dir / "metadata.json"
            shutil.copy(meta_source, meta_dest)
            packaged_files['metadata'] = {
                'path': str(meta_dest.relative_to(self.output_dir))
            }
        
        # Create model info
        model_pkg_info = {
            'name': model_name,
            'type': model_type,
            'version': '1.0.0',
            'accuracy': model_info.get('accuracy', 0),
            'f1_score': model_info.get('f1_score', 0),
            'inference_time_ms': model_info.get('inference_time_ms', 0),
            'files': packaged_files,
            'packaged_at': time.strftime('%Y-%m-%d %H:%M:%S')
        }
        
        # Save model info
        info_path = dest_dir / "package_info.json"
        with open(info_path, 'w') as f:
            json.dump(model_pkg_info, f, indent=2)
        
        self.package_manifest['models'][model_name] = model_pkg_info
        
        return model_pkg_info
    
    def save_manifest(self):
        """Save package manifest"""
        manifest_path = self.output_dir / "manifest.json"
        self.package_manifest['created_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
        with open(manifest_path, 'w') as f:
            json.dump(self.package_manifest, f, indent=2)
        return manifest_path

packager = ModelPackager(MODELS_DIR, BACKEND_DIR)
print("✓ Model Packager initialized")

## 3. Package Models

In [ ]:
# Load model registry
registry_path = MODELS_DIR / "model_registry.json"

if registry_path.exists():
    with open(registry_path) as f:
        registry = json.load(f)
    print(f"✓ Loaded {len(registry.get('models', {}))} models")
else:
    registry = {'models': {}}
    print("⚠ No registry found")

# Package each model
print("\nPackaging models for backend...\n")

for model_name, model_info in registry.get('models', {}).items():
    print(f"  Packaging: {model_name}")
    try:
        pkg_info = packager.package_model(model_name, model_info)
        print(f"    ✓ Files: {len(pkg_info['files'])}")
        print(f"    ✓ Version: {pkg_info['version']}")
    except Exception as e:
        print(f"    ⚠ Error: {e}")

# Save manifest
manifest_path = packager.save_manifest()
print(f"\n✓ Manifest saved to: {manifest_path}")

## 4. Generate Backend Integration Code

In [ ]:
# Generate Python inference module for backend
inference_module = '''
"""\nCyberForge ML Inference Module\nBackend integration for mlService.js\nIncludes /api/threats/detect and /api/analysis/explain endpoints.\n"""\n\nimport json\nimport time\nimport re\nimport joblib\nimport numpy as np\nfrom pathlib import Path\nfrom datetime import datetime\nfrom typing import Dict, List, Any, Optional\n\n\nclass CyberForgeInference:\n    """ML inference service for CyberForge backend."""\n\n    def __init__(self, models_dir: str):\n        self.models_dir = Path(models_dir)\n        self.loaded_models = {}\n        self.manifest = self._load_manifest()\n        self.threat_type_pipeline = None\n        self.risk_score_pipeline = None\n        self._load_threat_models()\n\n    def _load_manifest(self) -> Dict:\n        manifest_path = self.models_dir / "manifest.json"\n        if manifest_path.exists():\n            with open(manifest_path) as f:\n                return json.load(f)\n        return {"models": {}}\n\n    def _load_threat_models(self):\n        threat_dir = self.models_dir / "threat_detection"\n        type_path = threat_dir / "threat_type_pipeline.pkl"\n        score_path = threat_dir / "risk_score_pipeline.pkl"\n        if type_path.exists():\n            self.threat_type_pipeline = joblib.load(type_path)\n        if score_path.exists():\n            self.risk_score_pipeline = joblib.load(score_path)\n\n    def load_model(self, model_name: str) -> bool:\n        if model_name in self.loaded_models:\n            return True\n        model_dir = self.models_dir / model_name\n        model_path = model_dir / "model.pkl"\n        scaler_path = model_dir / "scaler.pkl"\n        if not model_path.exists():\n            return False\n        self.loaded_models[model_name] = {\n            "model": joblib.load(model_path),\n            "scaler": joblib.load(scaler_path) if scaler_path.exists() else None\n        }\n        return True\n\n    def predict(self, model_name: str, features: Dict) -> Dict:\n        if not self.load_model(model_name):\n            return {"error": f"Model not found: {model_name}"}\n        model_data = self.loaded_models[model_name]\n        model = model_data["model"]\n        scaler = model_data["scaler"]\n        X = np.array([list(features.values())])\n        if scaler:\n            X = scaler.transform(X)\n        start_time = time.time()\n        prediction = int(model.predict(X)[0])\n        inference_time = (time.time() - start_time) * 1000\n        confidence = 0.5\n        if hasattr(model, "predict_proba"):\n            proba = model.predict_proba(X)[0]\n            confidence = float(max(proba))\n        risk_level = "critical" if confidence >= 0.9 else "high" if confidence >= 0.7 else "medium" if confidence >= 0.5 else "low"\n        return {\n            "prediction": prediction,\n            "confidence": confidence,\n            "risk_level": risk_level,\n            "model_name": model_name,\n            "inference_time_ms": inference_time\n        }\n\n    # ── Threat Detection (matches /api/threats/detect contract) ──\n    THREAT_KEYWORDS = {\n        'phishing': (['phishing', 'credential', 'login', 'fake', 'spoof'], 25),\n        'malware': (['malware', 'trojan', 'ransomware', 'payload', 'exploit'], 30),\n        'suspicious': (['suspicious', 'obfuscated', 'redirect', 'tracking'], 15),\n        'crypto': (['cryptominer', 'mining', 'coinhive'], 20),\n        'botnet': (['botnet', 'c2', 'command and control', 'beacon'], 25),\n    }\n\n    def detect_threat(self, text: str, metadata: dict = None) -> dict:\n        metadata = metadata or {}\n        original_risk = float(metadata.get("riskScore", 0))\n        text_lower = text.lower()\n        indicators = []\n        risk_score = original_risk\n        threat_type = "benign"\n\n        # ML pipeline prediction (if available)\n        if self.threat_type_pipeline:\n            threat_type = self.threat_type_pipeline.predict([text])[0]\n            proba = self.threat_type_pipeline.predict_proba([text])[0]\n            ml_confidence = float(max(proba))\n        else:\n            ml_confidence = 0.5\n\n        if self.risk_score_pipeline:\n            risk_score = float(self.risk_score_pipeline.predict([text])[0])\n\n        # Heuristic enrichment\n        for cat, (keywords, weight) in self.THREAT_KEYWORDS.items():\n            hits = [kw for kw in keywords if kw in text_lower]\n            if hits:\n                indicators.extend([f'{cat}:{kw}' for kw in hits])\n                risk_score = min(risk_score + weight, 100)\n\n        if "missing security headers" in text_lower:\n            indicators.append('missing_headers')\n            risk_score = min(risk_score + 10, 100)\n        if "https: no" in text_lower:\n            indicators.append('no_https')\n            risk_score = min(risk_score + 15, 100)\n\n        confidence = max(ml_confidence, 0.55 + min(len(indicators) * 0.05, 0.35))\n\n        return {\n            "risk_score": round(min(risk_score, 100), 1),\n            "confidence": round(confidence, 2),\n            "threat_type": threat_type,\n            "indicators": indicators or ["no_signals"],\n            "details": {\n                "original_risk_score": original_risk,\n                "url": metadata.get("url", ""),\n                "analysis_method": "ml+heuristic" if self.threat_type_pipeline else "heuristic",\n                "analyzed_at": datetime.utcnow().isoformat()\n            }\n        }\n\n    # ── Analysis Explanation (matches /api/analysis/explain contract) ──\n    REC_MAP = {\n        'phishing': ['Block the domain at firewall/DNS', 'Alert users', 'Check credential stores'],\n        'malware': ['Quarantine endpoints', 'Scan with updated AV', 'Review lateral movement'],\n        'suspicious': ['Add to watchlist', 'Enable enhanced logging', 'Review security headers'],\n        'crypto': ['Block mining scripts via CSP', 'Scan endpoints', 'Review CPU usage'],\n        'botnet': ['Block C2 IPs/domains', 'Scan for persistence', 'Review DNS beaconing'],\n    }\n\n    def explain_threat(self, threat_data: dict) -> dict:\n        category = threat_data.get("category", "unknown")\n        threat_type = threat_data.get("threatType", category)\n        confidence = threat_data.get("confidence", 0)\n        risk_score = threat_data.get("riskScore", 0)\n        indicators = threat_data.get("indicators", [])\n\n        recs = self.REC_MAP.get(threat_type, self.REC_MAP.get(category, [\n            'Monitor the source', 'Review network traffic', 'Update threat feeds'\n        ]))\n\n        summary = (\n            f"{threat_type.replace('_', ' ').title()} threat detected "\n            f"with {round(confidence * 100)}% confidence (risk {risk_score}/100). "\n            f"{len(indicators)} indicator(s) triggered."\n        )\n        technical = (\n            f"Type: {threat_type} | Risk: {risk_score}/100 | "\n            f"Confidence: {round(confidence * 100)}% | "\n            f"Indicators: {', '.join(indicators[:5])}"\n        )\n\n        return {\n            "summary": summary,\n            "recommendations": recs[:5],\n            "technical_details": technical,\n            "timestamp": datetime.utcnow().isoformat()\n        }\n\n    def list_models(self) -> List[str]:\n        return list(self.manifest.get("models", {}).keys())\n\n    def get_model_info(self, model_name: str) -> Dict:\n        return self.manifest.get("models", {}).get(model_name, {})\n\n\n# ── FastAPI integration ──\ndef create_api(models_dir: str = '.'):\n    from fastapi import FastAPI, HTTPException\n    from pydantic import BaseModel\n\n    app = FastAPI(title="CyberForge ML API", version="1.0.0")\n    inference = CyberForgeInference(models_dir)\n\n    class PredictRequest(BaseModel):\n        model_name: str\n        features: Dict\n\n    class ThreatDetectRequest(BaseModel):\n        text: str\n        metadata: Optional[Dict] = {}\n\n    class ThreatExplainRequest(BaseModel):\n        threat_data: Dict\n\n    @app.post("/predict")\n    async def predict(request: PredictRequest):\n        result = inference.predict(request.model_name, request.features)\n        if "error" in result:\n            raise HTTPException(status_code=404, detail=result["error"])\n        return result\n\n    @app.post("/api/threats/detect")\n    async def detect_threat(request: ThreatDetectRequest):\n        return inference.detect_threat(request.text, request.metadata)\n\n    @app.post("/api/analysis/explain")\n    async def explain_threat(request: ThreatExplainRequest):\n        return inference.explain_threat(request.threat_data)\n\n    @app.get("/models")\n    async def list_models():\n        return {"models": inference.list_models()}\n\n    @app.get("/models/{model_name}")\n    async def get_model_info(model_name: str):\n        info = inference.get_model_info(model_name)\n        if not info:\n            raise HTTPException(status_code=404, detail="Model not found")\n        return info\n\n    @app.get("/health")\n    async def health():\n        return {\n            "status": "healthy",\n            "models_loaded": len(inference.loaded_models),\n            "threat_model_ready": inference.threat_type_pipeline is not None,\n            "timestamp": datetime.utcnow().isoformat()\n        }\n\n    return app\n\n\n# Module-level app for uvicorn\napp = create_api('.')\n\n\nif __name__ == "__main__":\n    import sys, uvicorn\n    models_dir = sys.argv[1] if len(sys.argv) > 1 else "."\n    app = create_api(models_dir)\n    uvicorn.run(app, host="0.0.0.0", port=8001)\n'''

inference_path = BACKEND_DIR / "inference.py"
with open(inference_path, 'w') as f:
    f.write(inference_module)

print(f"✓ Inference module saved to: {inference_path}")
print("  Endpoints: /predict, /api/threats/detect, /api/analysis/explain, /models, /health")


In [ ]:
# Generate JS client matching mlServiceClient.js contract
js_client = '''
/**\n * CyberForge ML Service Client — generated by notebook 06\n * Compatible with backend/src/services/mlServiceClient.js contract\n */\n\nconst axios = require('axios');\n\nclass CyberForgeMLClient {\n  constructor(baseUrl = 'http://localhost:8001') {\n    this.baseUrl = baseUrl;\n    this.client = axios.create({ baseURL: baseUrl, timeout: 30000 });\n  }\n\n  async healthCheck() {\n    const { data } = await this.client.get('/health');\n    return data;\n  }\n\n  /**\n   * Classify threat from evidence text.\n   * Contract: POST /api/threats/detect\n   * Called by: mlServiceClient.classifyThreat()\n   */\n  async classifyThreat(evidenceText, metadata = {}) {\n    const { data } = await this.client.post('/api/threats/detect', {\n      text: evidenceText,\n      metadata\n    });\n    return data;\n  }\n\n  /**\n   * Get threat explanation.\n   * Contract: POST /api/analysis/explain\n   * Called by: mlServiceClient.getExplanation()\n   */\n  async getExplanation(threatData) {\n    const { data } = await this.client.post('/api/analysis/explain', {\n      threat_data: threatData\n    });\n    return data;\n  }\n\n  /**\n   * Generic model prediction.\n   * Contract: POST /predict\n   */\n  async predict(modelName, features) {\n    const { data } = await this.client.post('/predict', {\n      model_name: modelName,\n      features\n    });\n    return data;\n  }\n\n  async listModels() {\n    const { data } = await this.client.get('/models');\n    return data.models;\n  }\n}\n\nmodule.exports = { CyberForgeMLClient };\n'''

js_client_path = BACKEND_DIR / "ml_client.js"
with open(js_client_path, 'w') as f:
    f.write(js_client)

print(f"✓ JS client saved to: {js_client_path}")
print("  Methods: classifyThreat(), getExplanation(), predict(), listModels(), healthCheck()")


## 5. Copy Agent Module

In [ ]:
# Copy agent module to backend package
agent_source = AGENT_DIR / "cyberforge_agent.py"
agent_config_source = AGENT_DIR / "agent_config.json"

if agent_source.exists():
    shutil.copy(agent_source, BACKEND_DIR / "cyberforge_agent.py")
    print(f"✓ Agent module copied")

if agent_config_source.exists():
    shutil.copy(agent_config_source, BACKEND_DIR / "agent_config.json")
    print(f"✓ Agent config copied")

## 6. Generate README

In [ ]:
readme_content = f"""
# CyberForge ML Backend Package

Production-ready ML models for CyberForge backend integration.

## Contents

- `inference.py` - Python inference module
- `ml_client.js` - JavaScript client for Node.js backend
- `cyberforge_agent.py` - Agent intelligence module
- `manifest.json` - Model registry and metadata
- `*/model.pkl` - Trained model files
- `*/scaler.pkl` - Feature scalers
- `*/metadata.json` - Model metadata

## Quick Start

### Python

```python
from inference import CyberForgeInference

inference = CyberForgeInference('./backend_package')
result = inference.predict('phishing_detection', {{'url_length': 50, ...}})
print(result)
```

### Node.js

```javascript
const CyberForgeMLClient = require('./ml_client');

const client = new CyberForgeMLClient('http://localhost:8001');
const result = await client.predict('phishing_detection', {{url_length: 50}});
console.log(result);
```

### FastAPI Server

```bash
pip install fastapi uvicorn
uvicorn inference:create_api --host 0.0.0.0 --port 8001
```

## API Contract

### Prediction Response

```json
{{
    "prediction": 0,
    "confidence": 0.95,
    "risk_level": "low",
    "model_name": "phishing_detection",
    "model_version": "1.0.0",
    "inference_time_ms": 2.5
}}
```

## Models Included

| Model | Type | Accuracy | F1 Score |
|-------|------|----------|----------|
"""

# Add model table
for model_name, model_info in packager.package_manifest.get('models', {}).items():
    readme_content += f"| {model_name} | {model_info.get('type', 'N/A')} | {model_info.get('accuracy', 0):.4f} | {model_info.get('f1_score', 0):.4f} |\n"

readme_content += f"""

## Version

- Package Version: 1.0.0
- Created: {time.strftime('%Y-%m-%d %H:%M:%S')}
"""

readme_path = BACKEND_DIR / "README.md"
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"✓ README saved to: {readme_path}")

## 7. Summary

In [ ]:
# List packaged files
packaged_files = list(BACKEND_DIR.rglob('*'))
total_size = sum(f.stat().st_size for f in packaged_files if f.is_file())

print("\n" + "=" * 60)
print("BACKEND INTEGRATION COMPLETE")
print("=" * 60)

print(f"""
📦 Backend Package:
   - Location: {BACKEND_DIR}
   - Files: {len([f for f in packaged_files if f.is_file()])}
   - Total size: {total_size / (1024*1024):.2f} MB

📁 Package Contents:""")

for f in sorted(BACKEND_DIR.iterdir()):
    if f.is_file():
        print(f"   - {f.name}")
    elif f.is_dir():
        print(f"   - {f.name}/")

print(f"""
🔌 Integration Points:
   - Python: inference.py (CyberForgeInference class)
   - Node.js: ml_client.js (CyberForgeMLClient class)
   - FastAPI: inference.py (create_api function)

Next step:
  → 07_deployment_artifacts.ipynb
""")
print("=" * 60)